In [ ]:
import torch
import numpy as np
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader
import pytorch_lightning as pl

# custom
from clearshape.invariants.ml.modules.invs_regressor import InvariantRegressor 
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from clearshape.dataset import FabwaveDataset
from clearshape.constants import PATHS
from clearshape.scaler.custom_scalers import LogScaler  


lo

# Dataset & DataLoader
test_dataset = FabwaveDataset(
    csv_file=PATHS.DATA_MODEL_INPUT / "test.csv",
    data_type="invariants",
    regression=True,
    scaler=LogScaler()
)
test_loader = DataLoader(test_dataset, batch_size=2000, shuffle=False)

# Modell laden
model = InvariantRegressor.load_from_checkpoint("/clear-shape/data/6_models/invariants-regressor-v3.ckpt")
model.eval();


ValueError: This LogScaler instance is not fitted yet.

In [14]:
# --- Inferenz ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

all_preds = []
all_targets = []

with torch.no_grad():
    for batch in test_loader:
        x, y, _ = batch
        x, y = x.to(device), y.to(device)
        preds = model(x)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y.cpu().numpy())

all_preds   = np.concatenate(all_preds,   axis=0)  # (N, 4)
all_targets = np.concatenate(all_targets, axis=0)  # (N, 4)

target_names = ['VOLUME', 'FACES', 'EDGES', 'VERTICES']

print("=" * 40)
print(f"{'Target':<12}  R²-Score")
print("=" * 40)

r2_scores = {}
for i, name in enumerate(target_names):
    r2 = r2_score(all_targets[:, i], all_preds[:, i])
    r2_scores[name] = r2
    print(f"{name:<12}  {r2:.4f}")

r2_overall = r2_score(all_targets, all_preds, multioutput='uniform_average')
print("-" * 40)
print(f"{'OVERALL':<12}  {r2_overall:.4f}")
print("=" * 40)

Target        R²-Score
VOLUME        -0.0186
FACES         -0.0843
EDGES         -0.0925
VERTICES      -0.0860
----------------------------------------
OVERALL       -0.0703
